In [76]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [77]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [78]:
import os, torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

ROOT = "/kaggle/input/datasets/valdivinosantiago/imagenettetvt320"
for d in os.listdir(ROOT):
    print(d)

val
test
train


In [79]:
class BasicBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1,use_downsample=False):
        super().__init__()

        self.relu = nn.ReLU(inplace=True)
        self.use_downsample = use_downsample

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.batch_norm1 = nn.BatchNorm2d(num_features = out_channels)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size = 3, stride=1, padding=1, bias=False)#<-not gonna add bias cuz gonna do batch norm
        self.batch_norm2 = nn.BatchNorm2d(num_features=out_channels)
        self.relu2 = nn.ReLU(inplace=True)

        if self.use_downsample:
            self.down_conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride = stride, padding=0, bias=False)
            self.down_bn = nn.BatchNorm2d(num_features = out_channels)


    def forward(self, x):
        identity = x
        x = self.relu1(self.batch_norm1(self.conv1(x)))
        x = self.batch_norm2(self.conv2(x))
        if self.use_downsample:
            identity = self.down_bn(self.down_conv(identity))

        x = x + identity
        x = self.relu(x)

        return x

In [80]:
class ResNet(nn.Module):

    def __init__(self, num_classes= 10):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7,stride=2, padding=3, bias = False)
        self.bn1 = nn.BatchNorm2d(num_features = 64)
        self.relu1 = nn.ReLU(inplace=True)

        self.pool1 = nn.MaxPool2d(kernel_size = 3,stride=2, padding=1)
        
        #stage_block (stage is for ones working at same resolution)
        self.block1_1 = BasicBlock(in_channels=64, out_channels=64,stride=1,use_downsample=False)
        self.block1_2 = BasicBlock(in_channels=64, out_channels=64,stride=1, use_downsample=False)
        
        self.block2_1 = BasicBlock(in_channels=64,out_channels=128,stride=2, use_downsample=True)
        self.block2_2 = BasicBlock(in_channels=128, out_channels=128, stride=1, use_downsample=False)
        
        self.block3_1 = BasicBlock(in_channels=128, out_channels=256, stride=2, use_downsample=True)
        self.block3_2 = BasicBlock(in_channels=256,out_channels=256, stride=1, use_downsample=False)
        
        self.block4_1 = BasicBlock(in_channels=256,out_channels=512, stride=2, use_downsample=True)
        self.block4_2 = BasicBlock(in_channels=512, out_channels=512, stride=1, use_downsample=False)
        
        self.avgpool = nn.AdaptiveAvgPool2d(output_size= (1,1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features = 512, out_features=num_classes)


    def forward(self,x):

        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))

        x = self.block1_1(x)
        x = self.block1_2(x)
        
        x = self.block2_1(x)
        x = self.block2_2(x)

        x = self.block3_1(x)
        x = self.block3_2(x)

        x = self.block4_1(x)
        x = self.block4_2(x)

        x= self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)

        return x

In [81]:
train_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.RandomCrop(320, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

val_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(320),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f"{ROOT}/train", transform=train_tf)
val_ds   = datasets.ImageFolder(f"{ROOT}/val",   transform=val_tf)
test_ds  = datasets.ImageFolder(f"{ROOT}/test",  transform=val_tf)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))
print("classes:", train_ds.classes)

train: 9469 val: 1309 test: 2616
classes: ['cassetePlayer', 'chainShaw', 'church', 'englishSpringer', 'frenchHorn', 'garbageTruck', 'gasPump', 'golfBall', 'parachute', 'tench']


In [82]:
device = "cuda"
model = ResNet(num_classes=10).to(device)

opti = torch.optim.SGD(params = model.parameters(),lr=.01, momentum = .9, weight_decay = 5e-4)
sched = torch.optim.lr_scheduler.StepLR(opti,step_size = 15,gamma = .1)
crit = nn.CrossEntropyLoss()
#f16 instead of f32
scaler = torch.amp.GradScaler("cuda")

best_acc = .0

for epoch in range(1,31):
    model.train()
    loss_sum = 0
    for x,y in train_dl:
        x,y = x.to(device,non_blocking=True), y.to(device, non_blocking=True)

        opti.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = crit(model(x),y)
        scaler.scale(loss).backward()
        scaler.step(opti)
        scaler.update()
        loss_sum += loss.item()
    sched.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total   += y.size(0)
    acc = correct / total
    print(f"epoch {epoch:02d}  loss {loss_sum/len(train_dl):.3f}  val_acc {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/kaggle/working/resnet_imagenette.pt")

print("best val_acc:", best_acc)

epoch 01  loss 1.885  val_acc 0.4293
epoch 02  loss 1.419  val_acc 0.5714
epoch 03  loss 1.201  val_acc 0.6050
epoch 04  loss 1.055  val_acc 0.6325
epoch 05  loss 0.942  val_acc 0.6646
epoch 06  loss 0.858  val_acc 0.7013
epoch 07  loss 0.788  val_acc 0.7517
epoch 08  loss 0.738  val_acc 0.7219
epoch 09  loss 0.699  val_acc 0.7387
epoch 10  loss 0.636  val_acc 0.6234
epoch 11  loss 0.621  val_acc 0.7647
epoch 12  loss 0.580  val_acc 0.7578
epoch 13  loss 0.557  val_acc 0.7189
epoch 14  loss 0.524  val_acc 0.7800
epoch 15  loss 0.498  val_acc 0.8189
epoch 16  loss 0.350  val_acc 0.8610
epoch 17  loss 0.299  val_acc 0.8625
epoch 18  loss 0.287  val_acc 0.8640
epoch 19  loss 0.270  val_acc 0.8663
epoch 20  loss 0.263  val_acc 0.8587
epoch 21  loss 0.260  val_acc 0.8724
epoch 22  loss 0.248  val_acc 0.8694
epoch 23  loss 0.235  val_acc 0.8686
epoch 24  loss 0.232  val_acc 0.8717
epoch 25  loss 0.233  val_acc 0.8701
epoch 26  loss 0.221  val_acc 0.8747
epoch 27  loss 0.205  val_acc 0.8648
e